# cerberus-neuro — NeuroPainting data exploration (S3 audit)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/01_data_exploration.ipynb)

Verify the public NeuroPainting dataset on the Cell Painting Gallery (`s3://cellpainting-gallery/cpg0038-tegtmeyer-neuropainting/`) against the project dossier. The dossier covers the descriptive layer (cohort, cell types, dyes, total size). This notebook resolves the gaps the dossier itself flags as unverified, plus the implementation details we need before designing the data pipeline.

**Verification punch list:**

1. Channel count: 5 vs 6. Resume says 5; the NeuroPainting protocol uses 6 dyes (Hoechst, MitoTracker, Phalloidin, ConA, SYTO 14, WGA). Confirm whether channels 3 (Phalloidin) and 4 (ConA) are stored separately in the gallery or pre-combined.
2. Brightfield channel: how labeled in `load_data.csv` filenames relative to fluorescence?
3. Image dimensions in pixels (not in the publication).
4. Fields of view per well. Paper implies 1; gallery may have more.
5. Cohort: 44 lines (22 control + 22 deletion) vs the unverified “plus 4 isogenic” resume claim.
6. Metadata schema: which `Metadata_*` columns encode cell type, cell line, and disease state.
7. File format, on-disk volume per plate, and what a Colab-Free-friendly subset looks like.

Stage A is listing + metadata-only (no images). Stage B is gated and downloads a single well to inspect channel mapping and dimensions.

Run `00_environment_smoke.ipynb` first to confirm the runtime is wired up.


## 1. Setup

In [157]:
!pip install -q "cerberus-neuro @ git+https://github.com/PatrickJReed/cerberus-neuro.git@main" imagecodecs

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [158]:
import os
from pathlib import Path

try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('HF login: OK (Colab secret)')
except Exception as e:
    print(f'HF login skipped ({type(e).__name__}); not required for the public S3 audit')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path('/content/drive/MyDrive/cerberus-neuro/cache/audit_v0')
except Exception:
    CACHE_DIR = Path.home() / '.cache' / 'cerberus-neuro' / 'audit_v0'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Cache dir: {CACHE_DIR}')

HF login: OK (Colab secret)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cache dir: /content/drive/MyDrive/cerberus-neuro/cache/audit_v0


## 2. Stage A — listing + metadata-only audit

Anonymous (unsigned) S3 access via `boto3`. Cell Painting Gallery is a public Registry of Open Data bucket; no AWS account required.

In [159]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = 'cellpainting-gallery'
ROOT_PREFIX = 'cpg0038-tegtmeyer-neuropainting/'
WORKSPACE_PREFIX = ROOT_PREFIX + 'broad/workspace/'
IMAGES_PREFIX    = ROOT_PREFIX + 'broad/images/'

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

def list_level(prefix, max_keys=2000):
    """Return (sub_prefixes, keys_at_this_level) using delimiter='/'."""
    paginator = s3.get_paginator('list_objects_v2')
    dirs, keys = [], []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix, Delimiter='/', PaginationConfig={'PageSize': max_keys}):
        dirs.extend(p['Prefix'] for p in page.get('CommonPrefixes', []))
        keys.extend((o['Key'], o['Size']) for o in page.get('Contents', []))
    return dirs, keys

def list_recursive(prefix, max_keys=None):
    """Flat list of all (key, size) under prefix. Pass max_keys=None for unbounded."""
    paginator = s3.get_paginator('list_objects_v2')
    cfg = {'PageSize': 1000}
    if max_keys is not None:
        cfg['MaxItems'] = max_keys
    out = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix, PaginationConfig=cfg):
        out.extend((o['Key'], o['Size']) for o in page.get('Contents', []))
    return out


### A1. Top-level prefix tree

Cell Painting Gallery datasets follow a `source_X / workspace / images` structure. Confirm what `cpg0038` actually has.

In [160]:
def show_tree(prefix, depth=2, indent=0):
    if indent > depth:
        return
    dirs, keys = list_level(prefix)
    pad = '  ' * indent
    for d in dirs:
        print(f'{pad}{d}')
        show_tree(d, depth=depth, indent=indent + 1)
    if indent == 0 and keys:
        print(f'{pad}({len(keys)} files at this level)')

show_tree(ROOT_PREFIX, depth=3)

cpg0038-tegtmeyer-neuropainting/broad/
  cpg0038-tegtmeyer-neuropainting/broad/images/
    cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/illum/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images_projected/
    cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_20x/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_20x/illum/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_20x/images/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_20x/images_projected/
    cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_63x/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_63x/illum/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_63x/images/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_63x/images_proj

### A2. Locate the metadata files

Cell Painting Gallery convention: per-plate `load_data.csv` / `load_data_with_illum.csv` lives under `workspace/load_data_csv/<batch>/<plate>/`, and dataset-level metadata (plate, well, compound) lives under `workspace/metadata/`.

In [161]:
import re

# Workspace holds metadata, load_data, analysis, profiles. Small relative to images/.
# Listing the whole root would hit hundreds of thousands of TIFFs first and starve the CSV scan.
workspace_keys = list_recursive(WORKSPACE_PREFIX)
print(f'{len(workspace_keys):,} objects under {WORKSPACE_PREFIX}')

csv_keys = [(k, sz) for k, sz in workspace_keys if k.endswith(('.csv', '.csv.gz', '.tsv', '.tsv.gz'))]
print(f'{len(csv_keys):,} CSV/TSV objects')
for k, sz in csv_keys[:30]:
    print(f'  {sz:>10}  {k}')
if len(csv_keys) > 30:
    print(f'  ... +{len(csv_keys) - 30} more')


243,995 objects under cpg0038-tegtmeyer-neuropainting/broad/workspace/
171,768 CSV/TSV objects
      308051  cpg0038-tegtmeyer-neuropainting/broad/workspace/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Cells.csv
      305936  cpg0038-tegtmeyer-neuropainting/broad/workspace/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Cytoplasm.csv
       48108  cpg0038-tegtmeyer-neuropainting/broad/workspace/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Experiment.csv
       61899  cpg0038-tegtmeyer-neuropainting/broad/workspace/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Image.csv
      306007  cpg0038-tegtmeyer-neuropainting/broad/workspace/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Nuclei.csv
      123907  cpg0038-tegtmeyer-neuropainting/broad/workspace/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-2/Cells.csv
      122788  cpg0038-tegtmeyer-neuropainting/broad/workspace

In [162]:
load_data_keys = [k for k, _ in csv_keys if 'load_data' in k.lower()]
metadata_keys  = [k for k, _ in csv_keys if '/metadata/' in k.lower()]
platemap_keys  = [k for k, _ in csv_keys if 'platemap' in k.lower() or 'plate_map' in k.lower()]

print('load_data.csv files:', len(load_data_keys))
for k in load_data_keys[:8]:
    print(' ', k)

print('\nmetadata/ files:', len(metadata_keys))
for k in metadata_keys[:20]:
    print(' ', k)

print('\nplate-map files:', len(platemap_keys))
for k in platemap_keys[:20]:
    print(' ', k)

load_data.csv files: 44
  cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_ASTROCYTES_1/PE_PP_Plate2/load_data.csv
  cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_ASTROCYTES_1/PE_PP_Plate2/load_data_projected.csv
  cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_ASTROCYTES_1/PE_PP_Plate2/load_data_projected_with_illum.csv
  cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_ASTROCYTES_1/Plate1_PE_PP96/load_data.csv
  cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_ASTROCYTES_1/Plate1_PE_PP96/load_data_projected.csv
  cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_ASTROCYTES_1/Plate1_PE_PP96/load_data_projected_with_illum.csv
  cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_NEURONS_2_20x/BR00132672/load_data.csv
  cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_NEURONS_2_20x/BR00132672/load_data_projected.csv

metadata/ files: 5
  cpg0038-tegtme

In [163]:
import pandas as pd

def s3_to_local(key):
    local = CACHE_DIR / key.replace('/', '__')
    if not local.exists():
        s3.download_file(BUCKET, key, str(local))
    return local

sample_load = load_data_keys[0] if load_data_keys else None
print(f'Inspecting: {sample_load}')
if sample_load:
    p = s3_to_local(sample_load)
    df = pd.read_csv(p)
    print(f'shape: {df.shape}')
    print('columns:')
    for c in df.columns:
        print(f'  {c}')
    print('\nfirst row:')
    print(df.iloc[0])

Inspecting: cpg0038-tegtmeyer-neuropainting/broad/workspace/load_data_csv/NCP_ASTROCYTES_1/PE_PP_Plate2/load_data.csv
shape: (2016, 32)
columns:
  URL_OrigDNA
  URL_OrigBrightfield
  URL_OrigMito
  URL_OrigAGP
  URL_OrigER
  URL_OrigRNA
  Metadata_Plate
  Metadata_Well
  Metadata_Site
  Metadata_AbsPositionZ
  Metadata_AbsTime
  Metadata_BinningX
  Metadata_BinningY
  Metadata_ChannelID
  Metadata_ChannelName
  Metadata_Col
  Metadata_ExposureTime
  Metadata_FieldID
  Metadata_ImageResolutionX
  Metadata_ImageResolutionY
  Metadata_ImageSizeX
  Metadata_ImageSizeY
  Metadata_MainEmissionWavelength
  Metadata_MainExcitationWavelength
  Metadata_MaxIntensity
  Metadata_ObjectiveMagnification
  Metadata_ObjectiveNA
  Metadata_PlaneID
  Metadata_PositionX
  Metadata_PositionY
  Metadata_PositionZ
  Metadata_Row

first row:
URL_OrigDNA                          s3://cellpainting-gallery/cpg0038-tegtmeyer-ne...
URL_OrigBrightfield                  s3://cellpainting-gallery/cpg0038-tegtmeyer-n

### A3. Channel naming — 5 vs 6, brightfield label

Enumerate every `FileName_*` / `PathName_*` column. Each fluorescence + brightfield channel will appear once. The stain abbreviation in the column name (e.g. `OrigDNA`, `OrigMito`, `OrigAGP`, `OrigER`, `OrigRNA`, `OrigBrightfield`) tells us how the gallery has organized the channels.

In [164]:
if sample_load:
    # CPG datasets either store one URL_ column per channel (full s3:// URI) or a
    # FileName_*/PathName_* pair. cpg0038 uses URL_*.
    url_cols  = [c for c in df.columns if c.startswith('URL_')]
    file_cols = [c for c in df.columns if c.startswith('FileName_')]
    path_cols = [c for c in df.columns if c.startswith('PathName_')]

    if url_cols:
        channel_cols = url_cols
        prefix = 'URL_'
        print(f'channel count (URL_ columns): {len(url_cols)}')
        for c in url_cols:
            print(f'  {c}  ->  {df[c].iloc[0]}')
    elif file_cols:
        channel_cols = file_cols
        prefix = 'FileName_'
        print(f'channel count (FileName_ columns): {len(file_cols)}')
        for c in file_cols:
            print(f'  {c}  ->  {df[c].iloc[0]}')
        print(f'\nPathName_ columns: {len(path_cols)}')
        for c in path_cols:
            print(f'  {c}  ->  {df[c].iloc[0]}')
    else:
        channel_cols, prefix = [], None
        print('No URL_ or FileName_ columns found')

    stains = [c[len(prefix):] for c in channel_cols] if prefix else []
    print(f'\nstain suffixes: {stains}')


channel count (URL_ columns): 6
  URL_OrigDNA  ->  s3://cellpainting-gallery/cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch6sk1fk1fl1.tiff
  URL_OrigBrightfield  ->  s3://cellpainting-gallery/cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch3sk1fk1fl1.tiff
  URL_OrigMito  ->  s3://cellpainting-gallery/cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch1sk1fk1fl1.tiff
  URL_OrigAGP  ->  s3://cellpainting-gallery/cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch2sk1fk1fl1.tiff
  URL_OrigER  ->  s3://cellpainting-gallery/cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r

### A4. Cohort verification — lines, plates, wells, cell types

Pull the dataset-level metadata (plate / well / line manifests) and compare to the dossier:
- 44 lines (22 control + 22 deletion); the resume’s extra “+4 isogenic” is unverified.
- 4 cell types: iPSC, NPC, Neuron, Astrocyte.
- 8 wells per cell line (per the publication).

In [165]:
for k in metadata_keys + platemap_keys:
    p = s3_to_local(k)
    try:
        m = pd.read_csv(p)
    except Exception as e:
        print(f'skip {k}: {e}')
        continue
    print(f'\n=== {k}  shape={m.shape} ===')
    print('columns:', list(m.columns))
    print(m.head(3))


=== cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_ASTROCYTES_1/barcode_platemap.csv  shape=(2, 2) ===
columns: ['Assay_Plate_Barcode', 'Plate_Map_Name']
  Assay_Plate_Barcode  Plate_Map_Name
0      Plate1_PE_PP96  Plate1_PE_PP96
1        PE_PP_Plate2    PE_PP_Plate2

=== cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_NEURONS_2_20x/barcode_platemap.csv  shape=(2, 2) ===
columns: ['Assay_Plate_Barcode', 'Plate_Map_Name']
  Assay_Plate_Barcode Plate_Map_Name
0          BR00132673     BR00132673
1          BR00132672     BR00132672

=== cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_NEURONS_2_63x/barcode_platemap.csv  shape=(2, 2) ===
columns: ['Assay_Plate_Barcode', 'Plate_Map_Name']
  Assay_Plate_Barcode Plate_Map_Name
0          BR00132673     BR00132673
1          BR00132672     BR00132672

=== cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_PROGENITORS_1/barcode_platemap.csv  shape=(1, 2) ===
columns: ['Assay_Plate_Barcode', 'Pl

In [166]:
# Cell line / cell type / disease state are not in load_data.csv (those Metadata_*
# columns are imaging-only). The barcode_platemap.csv files just map plate barcodes
# to plate-map names. Hunt for the actual per-well biology in three places:
#   (a) workspace/metadata_harmonized/  -- often holds normalized per-well manifests
#   (b) any *platemap*.txt/.csv referenced by Plate_Map_Name (besides barcode_platemap)
#   (c) workspace/publication_data/

mh_prefix  = WORKSPACE_PREFIX + 'metadata_harmonized/'
pub_prefix = WORKSPACE_PREFIX + 'publication_data/'

mh_keys  = list_recursive(mh_prefix)
pub_keys = list_recursive(pub_prefix)

print(f'metadata_harmonized: {len(mh_keys)} objects')
for k, sz in mh_keys[:30]:
    print(f'  {sz:>10}  {k}')

print(f'\npublication_data: {len(pub_keys)} objects')
for k, sz in pub_keys[:30]:
    print(f'  {sz:>10}  {k}')

other_platemap = [k for k, _ in workspace_keys
                  if ('platemap' in k.lower() or 'plate_map' in k.lower())
                  and not k.endswith('barcode_platemap.csv')]
print(f'\nother platemap-ish files: {len(other_platemap)}')
for k in other_platemap[:30]:
    print(f'  {k}')


metadata_harmonized: 1 objects
   130600761  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata_harmonized/cpg0038-tegtmeyer-neuropainting_draft.csv

publication_data: 120246 objects
       54922  cpg0038-tegtmeyer-neuropainting/broad/workspace/publication_data/2025_Haghighi_McPhie/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Cells.csv
         597  cpg0038-tegtmeyer-neuropainting/broad/workspace/publication_data/2025_Haghighi_McPhie/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Cytoplasm.csv
       19495  cpg0038-tegtmeyer-neuropainting/broad/workspace/publication_data/2025_Haghighi_McPhie/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Experiment.csv
       10105  cpg0038-tegtmeyer-neuropainting/broad/workspace/publication_data/2025_Haghighi_McPhie/analysis/NCP_ASTROCYTES_1/PE_PP_Plate2/analysis/PE_PP_Plate2-A01-1/Image.csv
        1147  cpg0038-tegtmeyer-neuropainting/broad/workspace/publication_data/2025_Haghighi_M

### A4b. Biological annotations — cell line, cell type, disease state

Two sources surfaced in A4: a 130 MB per-well harmonized manifest and per-plate platemap text files. Resolve the missing labels against both.

In [167]:
# (a) Master per-well manifest in metadata_harmonized/.
mh_csv = mh_keys[0][0]
mh_local = s3_to_local(mh_csv)
print(f'{mh_csv}  ({mh_local.stat().st_size/1e6:.1f} MB)')

mh_df = pd.read_csv(mh_local, low_memory=False)
print(f'shape: {mh_df.shape}')
print('columns:')
for c in mh_df.columns:
    print(f'  {c}')

# Resolve biological columns by pattern.
mh_cols = list(mh_df.columns)
mh_line     = first_match(mh_cols, r'^Metadata_line$', r'cell.?line', r'donor', r'subject', r'cell_id', r'\bline\b')
mh_celltype = first_match(mh_cols, r'cell.?type', r'differentiation', r'stage')
mh_disease  = first_match(mh_cols, r'genotype', r'disease', r'22q', r'\bcondition\b', r'\bgroup\b')
mh_plate    = first_match(mh_cols, r'^Metadata_Plate$', r'plate')
mh_well     = first_match(mh_cols, r'^Metadata_Well$', r'well')

print()
for label, c in [('cell line', mh_line), ('cell type', mh_celltype), ('disease state', mh_disease),
                 ('plate', mh_plate), ('well', mh_well)]:
    if c:
        vals = sorted(mh_df[c].dropna().astype(str).unique())
        print(f'{label:14s} <- {c:30s}  n_unique={len(vals)}')
        if len(vals) <= 60:
            print('   values:', vals)
    else:
        print(f'{label:14s} <- NOT FOUND in metadata_harmonized')

# Cross-tab: cell type x disease state if both resolved.
if mh_celltype and mh_disease:
    print('\ncell type x disease state (per-row counts, includes site/field replicates):')
    print(mh_df.groupby([mh_celltype, mh_disease]).size().unstack(fill_value=0))


cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata_harmonized/cpg0038-tegtmeyer-neuropainting_draft.csv  (130.6 MB)
shape: (307584, 44)
columns:
  Plate
  Well
  Site
  Binning
  Column
  Image_Size_X
  Image_Size_Y
  Maximum_Intensity
  Objective_Magnification
  Objective_NA
  Position_Z
  Row
  Batch
  User_Name
  File Path
  Label
  File Name
  Cell_Line
  Perturbation_Mechanism
  Cell_Origin
  Year_Imaged
  Microscope
  CP_Version
  Source
  Perturbation_Class
  Concentration
  Timepoint
  PubChem_CId
  Labeled_Molecule
  Solvent
  Modality
  Label_Mechanism
  Treatment
  Broad_Sample
  Cotreatment
  Pert_IName
  Investigator_Cell_Name
  Labeling_Reagent
  Cell_Type
  Treatment_Alternate_Name
  Perturbation_Category
  Smiles
  InChIKey
  Labeled_Structure

cell line      <- Cell_Line                       n_unique=4
   values: ['astro', 'neuron', 'progen', 'stem']
cell type      <- Cell_Type                       n_unique=0
   values: []
disease state  <- NOT FOUND in metadat

In [168]:
# (b) Inspect one platemap.txt to see how the per-plate well-to-line map is structured.
pm_key = 'cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_NEURONS_2_20x/platemap/BR00132672.txt'
pm_local = s3_to_local(pm_key)
pm = pd.read_csv(pm_local, sep='\t')
print(f'{pm_key}\nshape: {pm.shape}')
print('columns:', list(pm.columns))
print(pm.head(10))


cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_NEURONS_2_20x/platemap/BR00132672.txt
shape: (384, 7)
columns: ['Metadata_Plate', 'Metadata_Well', 'Metadata_cell_type', 'Metadata_line_ID', 'Metadata_line_condition', 'Metadata_line_source', 'Metadata_plating_density']
  Metadata_Plate Metadata_Well Metadata_cell_type  Metadata_line_ID  \
0     BR00132672           A01             neuron                 1   
1     BR00132672           A02             neuron                 1   
2     BR00132672           A03             neuron                30   
3     BR00132672           A04             neuron                30   
4     BR00132672           A05             neuron                 9   
5     BR00132672           A06             neuron                 9   
6     BR00132672           A07             neuron                26   
7     BR00132672           A08             neuron                26   
8     BR00132672           A09             neuron                 4   
9     BR0

### A4c. Cohort verification from aggregated platemap.txt files

The harmonized CSV's `Cell_Line` column actually encodes cell type (astro / neuron / progen / stem); `Cell_Type` is empty. The per-plate platemap.txt files are the gold source for cell-line ID, disease state, and isogenic status. Aggregate all eight, then check the 44-line cohort and the resume's "+4 isogenic" claim.

In [169]:
# Aggregate all per-plate platemap.txt files. Eight files (5 batches; neurons appear
# twice because the same physical plates were imaged at 20x and 63x).
pm_keys = sorted({k for k, _ in workspace_keys
                  if k.endswith('.txt') and '/platemap/' in k})
print(f'platemap.txt files: {len(pm_keys)}')
for k in pm_keys:
    print(f'  {k}')

frames = []
for k in pm_keys:
    p = s3_to_local(k)
    d = pd.read_csv(p, sep='\t')
    d['_platemap_key'] = k
    frames.append(d)
pmap = pd.concat(frames, ignore_index=True)
print(f'\naggregated: {pmap.shape}')
print('columns:', list(pmap.columns))

# Drop the duplicate neurons rows (same physical plate appears under both 20x and 63x).
pmap_unique = pmap.drop_duplicates(subset=['Metadata_Plate', 'Metadata_Well', 'Metadata_line_ID'])
print(f'after de-dup on (Plate, Well, line_ID): {pmap_unique.shape}')

# Cohort facts.
print('\ncell types:', sorted(pmap_unique['Metadata_cell_type'].dropna().unique()))
print('line conditions:', sorted(pmap_unique['Metadata_line_condition'].dropna().unique()))
print('line sources:',    sorted(pmap_unique['Metadata_line_source'].dropna().unique()))

print(f'\nunique Metadata_line_ID: {pmap_unique["Metadata_line_ID"].nunique()}')

# Lines x condition (each line should be entirely control OR deletion).
line_cond = pmap_unique.groupby('Metadata_line_ID')['Metadata_line_condition'].agg(lambda s: sorted(set(s)))
print('\nline_ID -> condition:')
print(line_cond.value_counts())

# Lines x source (each line either human or one of the isogenic flavors).
line_src = pmap_unique.groupby('Metadata_line_ID')['Metadata_line_source'].agg(lambda s: sorted(set(s)))
print('\nline_ID -> source:')
print(line_src.value_counts())

# Cohort split by source (human vs isogenic).
src_per_line = pmap_unique.groupby('Metadata_line_ID')['Metadata_line_source'].first()
cond_per_line = pmap_unique.groupby('Metadata_line_ID')['Metadata_line_condition'].first()

cohort = pd.DataFrame({'condition': cond_per_line, 'source': src_per_line})
print('\ncohort breakdown (line counts):')
print(cohort.groupby(['source', 'condition']).size())

# Wells per line per plate (publication says 8).
wpp = pmap_unique.groupby(['Metadata_Plate', 'Metadata_line_ID']).size()
print('\nwells-per-(plate, line) summary:')
print(wpp.describe().round(2))


platemap.txt files: 8
  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_ASTROCYTES_1/platemap/PE_PP_Plate2.txt
  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_ASTROCYTES_1/platemap/Plate1_PE_PP96.txt
  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_NEURONS_2_20x/platemap/BR00132672.txt
  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_NEURONS_2_20x/platemap/BR00132673.txt
  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_NEURONS_2_63x/platemap/BR00132672.txt
  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_NEURONS_2_63x/platemap/BR00132673.txt
  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_PROGENITORS_1/platemap/BR00127194.txt
  cpg0038-tegtmeyer-neuropainting/broad/workspace/metadata/NCP_STEM_1/platemap/BR_NCP_STEM_1.txt

aggregated: (2492, 8)
columns: ['Metadata_Plate', 'Metadata_Well', 'Metadata_cell_type', 'Metadata_line_ID', 'Metadata_line_condition', 'Metadata_line_source', 'Metadata

In [170]:
# Aggregate per-plate load_data.csv files to count plates / wells / sites / lines / cell types.
# Cell-type and disease-state encoding may live in Metadata_* columns of load_data, in a separate
# platemap, or in a well-level metadata CSV. Surface whatever is present.

frames = []
for k in load_data_keys:
    p = s3_to_local(k)
    try:
        d = pd.read_csv(p, usecols=lambda c: c.startswith('Metadata_') or c == 'Metadata_Site')
    except ValueError:
        d = pd.read_csv(p)
        d = d[[c for c in d.columns if c.startswith('Metadata_')]]
    d['_load_data_key'] = k
    frames.append(d)

meta = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f'rows across load_data files: {len(meta)}')
print('Metadata_* columns observed:')
for c in [c for c in meta.columns if c.startswith('Metadata_')]:
    n = meta[c].nunique(dropna=True)
    sample = sorted(meta[c].dropna().astype(str).unique())[:6]
    print(f'  {c:40s} unique={n:5d}  e.g. {sample}')

rows across load_data files: 117104
Metadata_* columns observed:
  Metadata_Plate                           unique=    6  e.g. ['BR00127194', 'BR00132672', 'BR00132673', 'BR_NCP_STEM_1', 'PE_PP_Plate2', 'Plate1_PE_PP96']
  Metadata_Well                            unique=  384  e.g. ['A01', 'A02', 'A03', 'A04', 'A05', 'A06']
  Metadata_Site                            unique=    9  e.g. ['1', '2', '3', '4', '5', '6']
  Metadata_AbsPositionZ                    unique= 6847  e.g. ['0.134278506', '0.134279206', '0.134279296', '0.1342794', '0.134280503', '0.134281203']
  Metadata_AbsTime                         unique=31104  e.g. ['2020-08-23T16:55:37.437-04:00', '2020-08-23T16:55:39.65-04:00', '2020-08-23T16:55:42.24-04:00', '2020-08-23T16:55:44.55-04:00', '2020-08-23T16:55:46.843-04:00', '2020-08-23T16:55:49.043-04:00']
  Metadata_BinningX                        unique=    1  e.g. ['1']
  Metadata_BinningY                        unique=    1  e.g. ['1']
  Metadata_ChannelID                

In [171]:
# Resolve cell-type / cell-line / disease-state columns by name pattern, then summarize.
import re as _re

def first_match(cols, *patterns):
    for p in patterns:
        for c in cols:
            if _re.search(p, c, _re.I):
                return c
    return None

cols = list(meta.columns)
col_plate     = first_match(cols, r'plate$', r'plate_id', r'plate')
col_well      = first_match(cols, r'well$', r'well_id', r'well')
col_site      = first_match(cols, r'site$', r'field')
col_line      = first_match(cols, r'line', r'donor', r'subject', r'cell_id')
col_celltype  = first_match(cols, r'cell.?type', r'differentiation', r'stage')
col_disease   = first_match(cols, r'genotype', r'disease', r'22q', r'condition', r'group')

for label, c in [('plate', col_plate), ('well', col_well), ('site/field', col_site),
                 ('cell line', col_line), ('cell type', col_celltype),
                 ('disease state', col_disease)]:
    if c:
        vals = sorted(meta[c].dropna().astype(str).unique())
        print(f'{label:14s} <- {c:30s}  n_unique={len(vals)}')
        if len(vals) <= 60:
            print('   values:', vals)
    else:
        print(f'{label:14s} <- NOT FOUND in load_data Metadata_* columns')

plate          <- Metadata_Plate                  n_unique=6
   values: ['BR00127194', 'BR00132672', 'BR00132673', 'BR_NCP_STEM_1', 'PE_PP_Plate2', 'Plate1_PE_PP96']
well           <- Metadata_Well                   n_unique=384
site/field     <- Metadata_Site                   n_unique=9
   values: ['1', '2', '3', '4', '5', '6', '7', '8', '9']
cell line      <- NOT FOUND in load_data Metadata_* columns
cell type      <- NOT FOUND in load_data Metadata_* columns
disease state  <- NOT FOUND in load_data Metadata_* columns


### A5. Volume math — what does a Colab-Free subset look like?

Tally TIFFs and total bytes for one plate to estimate per-plate footprint, then extrapolate.

In [172]:
# Per-batch image enumeration. Listing each batch separately keeps each request
# bounded and gives us per-batch counts for scoped-subset planning.
BATCHES = [
    'NCP_ASTROCYTES_1',
    'NCP_NEURONS_2_20x',
    'NCP_NEURONS_2_63x',
    'NCP_PROGENITORS_1',
    'NCP_STEM_1',
]

per_batch = {}
for b in BATCHES:
    keys = list_recursive(f'{IMAGES_PREFIX}{b}/images/')
    tiffs = [(k, sz) for k, sz in keys if k.lower().endswith(('.tif', '.tiff'))]
    n_tiff = len(tiffs)
    gb     = sum(sz for _, sz in tiffs) / 1e9
    per_batch[b] = {'n_tiff': n_tiff, 'gb': gb, 'n_objects': len(keys), 'tiffs': tiffs}
    print(f'{b:25s}  {n_tiff:>7,} tiffs  {gb:>7.1f} GB  ({len(keys):,} total objs)')

total_tiff = sum(v['n_tiff'] for v in per_batch.values())
total_gb   = sum(v['gb']     for v in per_batch.values())
print(f'\ntotal: {total_tiff:,} TIFFs, {total_gb:,.1f} GB')


NCP_ASTROCYTES_1            24,192 tiffs    206.3 GB  (24,198 total objs)
NCP_NEURONS_2_20x           41,472 tiffs    352.8 GB  (41,478 total objs)
NCP_NEURONS_2_63x           79,488 tiffs    405.8 GB  (79,494 total objs)
NCP_PROGENITORS_1           20,736 tiffs    187.0 GB  (20,739 total objs)
NCP_STEM_1                  20,736 tiffs    196.2 GB  (20,737 total objs)

total: 186,624 TIFFs, 1,348.2 GB


## 3. Stage B — single-well image inspection (gated)

Gated behind `RUN_STAGE_B = True`. Downloads ~50–100 MB for one well and renders each channel.

Resolves: image dimensions in pixels, dtype / bit depth, fields-of-view per well, and visual confirmation of the channel-to-dye mapping inferred in A3.

In [173]:
RUN_STAGE_B = True  # flip to False to skip the single-well download

if RUN_STAGE_B and sample_load:
    one = df.iloc[0]
    url_cols  = [c for c in df.columns if c.startswith('URL_')]
    file_cols = [c for c in df.columns if c.startswith('FileName_')]
    pairs = []
    if url_cols:
        # URL_* columns hold full s3:// URIs.
        for uc in url_cols:
            stain = uc[len('URL_'):]
            url = str(one[uc])
            scheme = f's3://{BUCKET}/'
            key = url[len(scheme):] if url.startswith(scheme) else url.lstrip('/')
            pairs.append((stain, key))
    elif file_cols:
        # Older CPG layout: FileName_* + PathName_* pairs.
        for fc in file_cols:
            stain = fc[len('FileName_'):]
            pc = f'PathName_{stain}'
            if pc in df.columns:
                path = str(one[pc])
                for pfx in (f's3://{BUCKET}/', '/'):
                    if path.startswith(pfx):
                        path = path[len(pfx):]
                key = f"{path.rstrip('/')}/{one[fc]}"
                pairs.append((stain, key))
    for stain, key in pairs:
        print(stain, '->', key)
else:
    print('Stage B not run. Set RUN_STAGE_B = True to download one well.')


OrigDNA -> cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch6sk1fk1fl1.tiff
OrigBrightfield -> cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch3sk1fk1fl1.tiff
OrigMito -> cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch1sk1fk1fl1.tiff
OrigAGP -> cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch2sk1fk1fl1.tiff
OrigER -> cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch5sk1fk1fl1.tiff
OrigRNA -> cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/PE_PP_Plate2__2022-08-07T11_34_05-Measurement/Images/r01c01f01p01-ch4sk1fk1fl1.tiff


In [ ]:
if RUN_STAGE_B and sample_load:
    # Decode with Pillow's libtiff backend, which handles LZW natively without
    # needing imagecodecs (and without the tifffile codec-registry caching pitfalls
    # that plagued earlier attempts in this Colab session).
    from PIL import Image
    import numpy as np
    import matplotlib.pyplot as plt

    well_dir = CACHE_DIR / 'well_sample'
    well_dir.mkdir(exist_ok=True)
    images = {}
    for stain, key in pairs:
        local = well_dir / Path(key).name
        if not local.exists():
            s3.download_file(BUCKET, key, str(local))
        with Image.open(local) as im:
            arr = np.array(im)
        images[stain] = arr
        print(f'{stain:20s} shape={arr.shape}  dtype={arr.dtype}  min={arr.min()}  max={arr.max()}')

    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, (stain, arr) in zip(axes, images.items()):
        a = arr.astype(np.float32)
        lo, hi = np.percentile(a, [1, 99])
        a = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
        ax.imshow(a, cmap='gray')
        ax.set_title(stain, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## 4. Audit summary

Capture the resolved facts for downstream pipeline design (`src/cerberus_neuro/data.py`).

In [ ]:
import json

if sample_load:
    detected_url  = [c for c in df.columns if c.startswith('URL_')]
    detected_file = [c for c in df.columns if c.startswith('FileName_')]
    detected_channels = detected_url or detected_file
else:
    detected_channels = []

cohort_summary = {}
if 'pmap_unique' in dir():
    src_per_line  = pmap_unique.groupby('Metadata_line_ID')['Metadata_line_source'].first()
    cond_per_line = pmap_unique.groupby('Metadata_line_ID')['Metadata_line_condition'].first()
    cohort_df = pd.DataFrame({'condition': cond_per_line, 'source': src_per_line})
    breakdown = [
        {'source': s, 'condition': c, 'n_lines': int(n)}
        for (s, c), n in cohort_df.groupby(['source', 'condition']).size().items()
    ]
    cohort_summary = {
        'n_unique_lines':  int(pmap_unique['Metadata_line_ID'].nunique()),
        'cell_types':      sorted(pmap_unique['Metadata_cell_type'].dropna().unique().tolist()),
        'line_conditions': sorted(pmap_unique['Metadata_line_condition'].dropna().unique().tolist()),
        'line_sources':    sorted(pmap_unique['Metadata_line_source'].dropna().unique().tolist()),
        'cohort_breakdown': breakdown,
    }

audit = {
    'bucket': BUCKET,
    'root_prefix': ROOT_PREFIX,
    'workspace_prefix': WORKSPACE_PREFIX,
    'images_prefix': IMAGES_PREFIX,
    'batches': list(per_batch.keys()),
    'per_batch_summary': {b: {'n_tiff': v['n_tiff'], 'gb': round(v['gb'], 1)} for b, v in per_batch.items()},
    'total_tiff_count': total_tiff,
    'total_tiff_gb':    round(total_gb, 1),
    'load_data_files':  len(load_data_keys),
    'metadata_files':   metadata_keys,
    'channel_columns':  detected_channels,
    'image_size_xy':    sorted(meta['Metadata_ImageSizeX'].dropna().unique().tolist()) if 'Metadata_ImageSizeX' in meta.columns else None,
    'sites_per_well':   sorted(meta['Metadata_Site'].dropna().unique().tolist()) if 'Metadata_Site' in meta.columns else None,
    'imaging_metadata_columns': {
        'plate': col_plate, 'well': col_well, 'site': col_site,
    },
    'biological_metadata_source': 'workspace/metadata/<batch>/platemap/<plate_barcode>.txt',
    'biological_metadata_columns': {
        'plate': 'Metadata_Plate',
        'well':  'Metadata_Well',
        'cell_type':       'Metadata_cell_type',
        'cell_line':       'Metadata_line_ID',
        'disease_state':   'Metadata_line_condition',
        'isogenic_status': 'Metadata_line_source',
    },
    'cohort': cohort_summary,
}
out = CACHE_DIR / 'audit_v0_findings.json'
out.write_text(json.dumps(audit, indent=2, default=str))
print(f'wrote {out}')
print(json.dumps(audit, indent=2, default=str))
